In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.chdir('/content/drive/MyDrive/BBAdvGen/Baselines')

!ls /content/drive/MyDrive/BBAdvGen/Baselines

print('Current directory:', os.getcwd())

Mounted at /content/drive
Autozoom-Attack  AutoZOOM.ipynb
Current directory: /content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines


In [2]:
# !git clone https://github.com/IBM/Autozoom-Attack

Cloning into 'Autozoom-Attack'...
remote: Enumerating objects: 58, done.
remote: Total 58 (delta 0), reused 0 (delta 0), pack-reused 58 (from 1)
Receiving objects: 100% (58/58), 10.53 MiB | 14.43 MiB/s, done.
Resolving deltas: 100% (32/32), done.


In [2]:
import os
os.chdir('/content/drive/MyDrive/BBAdvGen/Baselines/Autozoom-Attack')

In [3]:
from pathlib import Path

root = Path("/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack")

for path in root.rglob("*.py"):

    text = path.read_text()

    new_text = text.replace(

        "tensorflow.contrib.keras.api.keras",

        "tensorflow.keras"

    )

    if new_text != text:

        path.write_text(new_text)

        print("Patched:", path)

Patched: /content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/Utils.py
Patched: /content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/setup_cifar.py
Patched: /content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/setup_codec.py
Patched: /content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/setup_inception.py
Patched: /content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/setup_mnist.py


In [5]:
!grep -R "tf\.app\|tf\.contrib\|tf\.Session\|tf\.placeholder\|tf\.reset_default_graph" \
"/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack"

/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/blackbox_attack.py:        self.assign_timg = tf.placeholder(tf.float32, self.image_shape)
/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/blackbox_attack.py:        self.assign_tlab = tf.placeholder(tf.float32, self.num_labels)
/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/blackbox_attack.py:        self.assign_const = tf.placeholder(tf.float32)
/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/blackbox_attack.py:        self.modifier = tf.placeholder(tf.float32, shape=(None,) + self.modifier_shape)
/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/blackbox_attack.py:        self.modifier = tf.placeholder(tf.float32, shape=(None,) + self.modifie

In [8]:
from pathlib import Path

ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/"
    "BBAdvGen/Baselines/Autozoom-Attack"
)

# ---------------------------------------------------------
# 1. Patch TF1 APIs → TF2 compatibility APIs
# ---------------------------------------------------------

replacements = {
    # TensorFlow graph APIs
    "tf.placeholder(": "tf.compat.v1.placeholder(",
    "tf.Session(": "tf.compat.v1.Session(",

    # tf.app.flags
    "tf.app.flags.FLAGS": "tf.compat.v1.app.flags.FLAGS",
    "tf.app.flags.DEFINE_string(": "tf.compat.v1.app.flags.DEFINE_string(",
    "tf.app.flags.DEFINE_integer(": "tf.compat.v1.app.flags.DEFINE_integer(",
    "tf.app.flags.DEFINE_float(": "tf.compat.v1.app.flags.DEFINE_float(",
    "tf.app.flags.DEFINE_boolean(": "tf.compat.v1.app.flags.DEFINE_boolean(",

    # tf.app.run
    "tf.app.run(": "tf.compat.v1.app.run(",

    # Other common TF1 APIs
    "tf.reset_default_graph(": "tf.compat.v1.reset_default_graph(",
    "tf.global_variables_initializer(": "tf.compat.v1.global_variables_initializer(",
    "tf.get_default_graph(": "tf.compat.v1.get_default_graph(",
    "tf.variable_scope(": "tf.compat.v1.variable_scope(",
    "tf.get_variable(": "tf.compat.v1.get_variable(",
}

# ---------------------------------------------------------
# 2. Patch tensorflow.contrib.keras imports
# ---------------------------------------------------------

contrib_replacements = {
    "tensorflow.contrib.keras.api.keras": "tensorflow.keras",
}

# ---------------------------------------------------------
# 3. Apply patches
# ---------------------------------------------------------

for path in ROOT.rglob("*.py"):
    text = path.read_text()
    original = text

    # TF1 API replacements
    for old, new in replacements.items():
        text = text.replace(old, new)

    # tensorflow.contrib.keras → tensorflow.keras
    for old, new in contrib_replacements.items():
        text = text.replace(old, new)

    if text != original:
        path.write_text(text)
        print("Patched:", path.relative_to(ROOT))

print("\nDone.")

Patched: blackbox_attack.py
Patched: main.py
Patched: setup_inception.py

Done.


In [10]:
!grep -R "tf\.placeholder\|tf\.Session\|tf\.app\|tensorflow\.contrib" -n \
"/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack" \
|| echo "No old APIs found."

No old APIs found.


In [11]:
from pathlib import Path

ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/"
    "BBAdvGen/Baselines/Autozoom-Attack"
)

replacements = {
    "tf.set_random_seed(": "tf.compat.v1.set_random_seed(",
    "tf.placeholder(": "tf.compat.v1.placeholder(",
    "tf.Session(": "tf.compat.v1.Session(",
    "tf.reset_default_graph(": "tf.compat.v1.reset_default_graph(",
    "tf.global_variables_initializer(": "tf.compat.v1.global_variables_initializer(",
    "tf.get_default_graph(": "tf.compat.v1.get_default_graph(",
    "tf.variable_scope(": "tf.compat.v1.variable_scope(",
    "tf.get_variable(": "tf.compat.v1.get_variable(",
    "tf.app.flags.FLAGS": "tf.compat.v1.app.flags.FLAGS",
    "tf.app.flags.DEFINE_string(": "tf.compat.v1.app.flags.DEFINE_string(",
    "tf.app.flags.DEFINE_integer(": "tf.compat.v1.app.flags.DEFINE_integer(",
    "tf.app.flags.DEFINE_float(": "tf.compat.v1.app.flags.DEFINE_float(",
    "tf.app.flags.DEFINE_boolean(": "tf.compat.v1.app.flags.DEFINE_boolean(",
    "tf.app.run(": "tf.compat.v1.app.run(",
}

for path in ROOT.rglob("*.py"):
    text = path.read_text()
    original = text

    for old, new in replacements.items():
        text = text.replace(old, new)

    if text != original:
        path.write_text(text)
        print("Patched:", path.relative_to(ROOT))

print("Done.")

Patched: main.py
Done.


In [14]:
print('Current directory:', os.getcwd())

Current directory: /content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack


In [15]:
!python setup_mnist.py

In [17]:
import os
import urllib.request

data_dir = "/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/data"
os.makedirs(data_dir, exist_ok=True)

base = "https://storage.googleapis.com/cvdf-datasets/mnist/"

files = [
    "train-images-idx3-ubyte.gz",
    "train-labels-idx1-ubyte.gz",
    "t10k-images-idx3-ubyte.gz",
    "t10k-labels-idx1-ubyte.gz",
]

for name in files:
    path = os.path.join(data_dir, name)

    if not os.path.exists(path):
        print("Downloading:", name)
        urllib.request.urlretrieve(base + name, path)
    else:
        print("Already exists:", name)

print("MNIST downloaded.")

Downloading: train-images-idx3-ubyte.gz
Downloading: train-labels-idx1-ubyte.gz
Downloading: t10k-images-idx3-ubyte.gz
Downloading: t10k-labels-idx1-ubyte.gz
MNIST downloaded.


In [19]:
!find "/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/models" \
-maxdepth 2 -type f -printf "%p\n"

/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/models/cifar
/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/models/mnist


In [20]:
!ls -lah "/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/models"

total 12M
-rw------- 1 root root 8.9M Sep  4 20:43 cifar
-rw------- 1 root root 2.5M Sep  4 20:43 mnist


In [21]:
!file "/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/models/mnist"

/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/models/mnist: Hierarchical Data Format (version 5) data


In [22]:
!head -c 20 "/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/models/mnist" | xxd

00000000: 8948 4446 0d0a 1a0a 0000 0000 0008 0800  .HDF............
00000010: 0400 1000                                ....


In [23]:
!ls -lh "/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/models/mnist"

-rw------- 1 root root 2.5M Sep  4 20:43 /content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/models/mnist


In [24]:
import h5py

path = "/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/models/mnist"

with h5py.File(path, "r") as f:
    print("Root keys:", list(f.keys()))
    print("Attributes:")
    for k, v in f.attrs.items():
        print(" ", k, v)

Root keys: ['model_weights', 'optimizer_weights']
Attributes:
  backend tensorflow
  keras_version 2.0.8
  model_config {"class_name": "Sequential", "config": [{"class_name": "Conv2D", "config": {"name": "conv2d_1", "trainable": true, "batch_input_shape": [null, 28, 28, 1], "dtype": "float32", "filters": 32, "kernel_size": [3, 3], "strides": [1, 1], "padding": "valid", "data_format": "channels_last", "dilation_rate": [1, 1], "activation": "linear", "use_bias": true, "kernel_initializer": {"class_name": "VarianceScaling", "config": {"scale": 1.0, "mode": "fan_avg", "distribution": "uniform", "seed": null}}, "bias_initializer": {"class_name": "Zeros", "config": {}}, "kernel_regularizer": null, "bias_regularizer": null, "activity_regularizer": null, "kernel_constraint": null, "bias_constraint": null}}, {"class_name": "Activation", "config": {"name": "activation_1", "trainable": true, "activation": "relu"}}, {"class_name": "Conv2D", "config": {"name": "conv2d_2", "trainable": true, "filter

In [25]:
from pathlib import Path
import shutil

ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/"
    "BBAdvGen/Baselines/Autozoom-Attack"
)

src = ROOT / "models" / "mnist"
dst = ROOT / "models" / "mnist.h5"

if not dst.exists():
    shutil.copy2(src, dst)

print("Model:", dst)

Model: /content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/models/mnist.h5


In [26]:
import tensorflow as tf
import h5py
import json
import numpy as np

tf.compat.v1.disable_eager_execution()

MODEL_PATH = (
    "/content/drive/.shortcut-targets-by-id/"
    "15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/"
    "BBAdvGen/Baselines/Autozoom-Attack/models/mnist"
)

# Read the legacy model configuration
with h5py.File(MODEL_PATH, "r") as f:
    model_config = f.attrs["model_config"]

    if isinstance(model_config, bytes):
        model_config = model_config.decode("utf-8")

print(model_config[:500])

{"class_name": "Sequential", "config": [{"class_name": "Conv2D", "config": {"name": "conv2d_1", "trainable": true, "batch_input_shape": [null, 28, 28, 1], "dtype": "float32", "filters": 32, "kernel_size": [3, 3], "strides": [1, 1], "padding": "valid", "data_format": "channels_last", "dilation_rate": [1, 1], "activation": "linear", "use_bias": true, "kernel_initializer": {"class_name": "VarianceScaling", "config": {"scale": 1.0, "mode": "fan_avg", "distribution": "uniform", "seed": null}}, "bias_


In [29]:
from pathlib import Path

ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/"
    "BBAdvGen/Baselines/Autozoom-Attack"
)

utils_path = ROOT / "Utils.py"

text = utils_path.read_text()

old = "pred_labels = np.argmax(model.model.predict(data.test_data), axis=1)"

new = """pred_logits = model.sess.run(
        model.model.output,
        feed_dict={
            model.model.input: data.test_data
        }
    )
    pred_labels = np.argmax(pred_logits, axis=1)"""

if old not in text:
    print("Old line not found!")
else:
    text = text.replace(old, new)
    utils_path.write_text(text)
    print("Patched Utils.py")

Patched Utils.py


In [27]:
from pathlib import Path

main_path = ROOT / "main.py"

text = main_path.read_text()

text = text.replace(
    'MNISTModel("models/mnist", sess, use_softmax=True)',
    'MNISTModel("models/mnist.h5", sess, use_softmax=True)'
)

main_path.write_text(text)

print("Patched main.py")

Patched main.py


In [31]:
from pathlib import Path

ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/"
    "BBAdvGen/Baselines/Autozoom-Attack"
)

!grep -R -n -A80 -B10 "class MNISTModel" "$ROOT"

/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/setup_mnist.py-50-        self.test_labels = extract_labels("data/t10k-labels-idx1-ubyte.gz", 10000)
/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/setup_mnist.py-51-        
/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/setup_mnist.py-52-        VALIDATION_SIZE = 5000
/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/setup_mnist.py-53-        
/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/setup_mnist.py-54-        self.validation_data = train_data[:VALIDATION_SIZE, :, :, :]
/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/setup_mnist.py-55-        self.validation_labels = train_labe

In [34]:
from pathlib import Path

ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/"
    "BBAdvGen/Baselines/Autozoom-Attack"
)

path = ROOT / "Utils.py"
text = path.read_text()

text = text.replace(
    """pred_logits = tf.compat.v1.get_default_session().run(
        model.predict(data.test_data)
    )""",
    """pred_logits = tf.compat.v1.get_default_session().run(
        model.predict(data.test_data)
    )"""
)

path.write_text(text)

7356

In [35]:
!sed -n '1,110p' setup_mnist.py

## setup_mnist.py -- mnist data and model loading code
##
## Copyright (C) 2016, Nicholas Carlini <nicholas@carlini.com>.
##
## Original copyright license follows.

import tensorflow as tf
import numpy as np
import os
import pickle
import gzip
import urllib.request

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.models import load_model

def extract_data(filename, num_images):
    with gzip.open(filename) as bytestream:
        bytestream.read(16)
        buf = bytestream.read(num_images*28*28)
        data = np.frombuffer(buf, dtype=np.uint8).astype(np.float32)
        data = (data / 255) - 0.5
        data = data.reshape(num_images, 28, 28, 1)
        return data

def extract_labels(filename, num_images):
    with gzip.open(filename) as bytestream:
        bytestream.read(8)
        buf = bytestream.read(1 * num_images)
        labels 

In [36]:
from pathlib import Path
import os

ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/"
    "BBAdvGen/Baselines/Autozoom-Attack"
)

os.chdir(ROOT)

print("Current directory:")
print(os.getcwd())

Current directory:
/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack


In [37]:
from pathlib import Path
import h5py

model_path = ROOT / "models" / "mnist"

print("Original:", model_path.exists())
print("mnist.h5:", (ROOT / "models" / "mnist.h5").exists())

Original: True
mnist.h5: True


In [38]:
from pathlib import Path

setup_code = r'''
# setup_mnist.py
#
# Legacy AutoZOOM MNIST loader adapted for modern
# TensorFlow/Keras running in TF1-compatible graph mode.

import tensorflow as tf

# Must be executed before constructing the model.
tf.compat.v1.disable_eager_execution()

import numpy as np
import os
import gzip
import urllib.request
import h5py

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D


def extract_data(filename, num_images):
    with gzip.open(filename) as bytestream:
        bytestream.read(16)

        buf = bytestream.read(num_images * 28 * 28)

        data = np.frombuffer(
            buf,
            dtype=np.uint8
        ).astype(np.float32)

        data = (data / 255.0) - 0.5

        data = data.reshape(
            num_images,
            28,
            28,
            1
        )

        return data


def extract_labels(filename, num_images):
    with gzip.open(filename) as bytestream:
        bytestream.read(8)

        buf = bytestream.read(num_images)

        labels = np.frombuffer(
            buf,
            dtype=np.uint8
        )

    return (
        np.arange(10) == labels[:, None]
    ).astype(np.float32)


class MNIST:

    def __init__(self):

        if not os.path.exists("data"):
            os.mkdir("data")

        files = [
            "train-images-idx3-ubyte.gz",
            "t10k-images-idx3-ubyte.gz",
            "train-labels-idx1-ubyte.gz",
            "t10k-labels-idx1-ubyte.gz"
        ]

        for name in files:

            path = os.path.join(
                "data",
                name
            )

            if not os.path.exists(path):

                urllib.request.urlretrieve(
                    "https://storage.googleapis.com/cvdf-datasets/mnist/" + name,
                    path
                )

        train_data = extract_data(
            "data/train-images-idx3-ubyte.gz",
            60000
        )

        train_labels = extract_labels(
            "data/train-labels-idx1-ubyte.gz",
            60000
        )

        self.test_data = extract_data(
            "data/t10k-images-idx3-ubyte.gz",
            10000
        )

        self.test_labels = extract_labels(
            "data/t10k-labels-idx1-ubyte.gz",
            10000
        )

        VALIDATION_SIZE = 5000

        self.validation_data = train_data[
            :VALIDATION_SIZE
        ]

        self.validation_labels = train_labels[
            :VALIDATION_SIZE
        ]

        self.train_data = train_data[
            VALIDATION_SIZE:
        ]

        self.train_labels = train_labels[
            VALIDATION_SIZE:
        ]


class MNISTModel:

    def __init__(
        self,
        restore=None,
        session=None,
        use_softmax=False
    ):

        self.num_channels = 1
        self.image_size = 28
        self.num_labels = 10

        if session is None:
            session = tf.compat.v1.get_default_session()

        if session is None:
            raise RuntimeError(
                "MNISTModel requires an active TF session."
            )

        self.session = session

        # ---------------------------------------------------------
        # Build exactly the original architecture
        # ---------------------------------------------------------

        self.model = Sequential(
            name="mnist_classifier"
        )

        self.model.add(
            Conv2D(
                32,
                (3, 3),
                input_shape=(28, 28, 1)
            )
        )

        self.model.add(
            Activation("relu")
        )

        self.model.add(
            Conv2D(32, (3, 3))
        )

        self.model.add(
            Activation("relu")
        )

        self.model.add(
            MaxPooling2D(
                pool_size=(2, 2)
            )
        )

        self.model.add(
            Conv2D(64, (3, 3))
        )

        self.model.add(
            Activation("relu")
        )

        self.model.add(
            Conv2D(64, (3, 3))
        )

        self.model.add(
            Activation("relu")
        )

        self.model.add(
            MaxPooling2D(
                pool_size=(2, 2)
            )
        )

        self.model.add(
            Flatten()
        )

        self.model.add(
            Dense(200)
        )

        self.model.add(
            Activation("relu")
        )

        self.model.add(
            Dense(200)
        )

        self.model.add(
            Activation("relu")
        )

        self.model.add(
            Dense(10)
        )

        if use_softmax:

            self.model.add(
                Activation("softmax")
            )

        # Force creation of all variables.
        dummy = tf.zeros(
            [1, 28, 28, 1],
            dtype=tf.float32
        )

        self.output_tensor = self.model(dummy)

        # ---------------------------------------------------------
        # Initialize variables FIRST.
        # ---------------------------------------------------------

        self.session.run(
            tf.compat.v1.global_variables_initializer()
        )

        # ---------------------------------------------------------
        # Load legacy Keras 2.x HDF5 weights manually.
        # ---------------------------------------------------------

        if restore:
            self._load_legacy_weights(restore)

    def _load_legacy_weights(self, filename):

        print(
            "Loading legacy HDF5 weights:",
            filename
        )

        with h5py.File(filename, "r") as f:

            if "model_weights" not in f:
                raise RuntimeError(
                    "HDF5 file does not contain model_weights."
                )

            root = f["model_weights"]

            # Keras 2.x stores layer names here.
            if "layer_names" in root.attrs:

                layer_names = [
                    n.decode("utf-8")
                    if isinstance(n, bytes)
                    else n
                    for n in root.attrs["layer_names"]
                ]

            else:

                layer_names = list(root.keys())

            assignments = []

            # -----------------------------------------------------
            # Read weights layer-by-layer.
            # -----------------------------------------------------

            for layer_name in layer_names:

                if layer_name not in root:
                    continue

                layer_group = root[layer_name]

                if "weight_names" not in layer_group.attrs:
                    continue

                weight_names = [
                    n.decode("utf-8")
                    if isinstance(n, bytes)
                    else n
                    for n in layer_group.attrs["weight_names"]
                ]

                # Find corresponding Keras layer.
                keras_layer = None

                for layer in self.model.layers:

                    if layer.name == layer_name:
                        keras_layer = layer
                        break

                # Keras 3 may rename layers, so if exact
                # name is unavailable, match by having weights.
                if keras_layer is None:
                    continue

                keras_weights = keras_layer.weights

                if len(keras_weights) != len(weight_names):
                    print(
                        "WARNING:",
                        layer_name,
                        "HDF5 weights:",
                        len(weight_names),
                        "Keras weights:",
                        len(keras_weights)
                    )

                    continue

                for variable, weight_name in zip(
                    keras_weights,
                    weight_names
                ):

                    dataset_name = weight_name.split("/")[-1]

                    # Usually dataset is directly under layer group.
                    if dataset_name in layer_group:
                        value = np.array(
                            layer_group[dataset_name]
                        )
                    elif weight_name in layer_group:
                        value = np.array(
                            layer_group[weight_name]
                        )
                    else:
                        # Search recursively.
                        value = None

                        def visitor(name, obj):
                            nonlocal value

                            if (
                                value is None
                                and isinstance(
                                    obj,
                                    h5py.Dataset
                                )
                                and (
                                    name.endswith(
                                        dataset_name
                                    )
                                    or name.endswith(
                                        weight_name
                                    )
                                )
                            ):
                                value = np.array(obj)

                        layer_group.visititems(visitor)

                        if value is None:
                            raise RuntimeError(
                                "Could not find HDF5 dataset "
                                + weight_name
                            )

                    if tuple(value.shape) != tuple(
                        variable.shape
                    ):
                        raise RuntimeError(
                            "Shape mismatch for "
                            + layer_name
                            + "/"
                            + weight_name
                            + ": HDF5="
                            + str(value.shape)
                            + " Keras="
                            + str(variable.shape)
                        )

                    assignments.append(
                        variable.assign(value)
                    )

            if not assignments:

                raise RuntimeError(
                    "No weights were loaded from "
                    + filename
                )

            # Execute ALL assignments in the same session.
            self.session.run(assignments)

        print(
            "Loaded",
            len(assignments),
            "weight tensors."
        )

    def predict(self, data):
        return self.model(data)
'''

(ROOT / "setup_mnist.py").write_text(
    setup_code
)

print("setup_mnist.py replaced.")

setup_mnist.py replaced.


In [39]:
import tensorflow as tf

tf.compat.v1.disable_eager_execution()
tf.compat.v1.reset_default_graph()

from setup_mnist import MNIST, MNISTModel

with tf.compat.v1.Session() as sess:

    data = MNIST()

    model = MNISTModel(
        restore="models/mnist.h5",
        session=sess,
        use_softmax=True
    )

    print("\nModel:")
    model.model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Loading legacy HDF5 weights: models/mnist.h5


RuntimeError: Shape mismatch for conv2d_1/conv2d_1/kernel:0: HDF5=(3, 3, 1, 32) Keras=(3, 3, 32, 32)

In [ ]:
import numpy as np

with tf.compat.v1.Session() as sess:

    data = MNIST()

    model = MNISTModel(
        restore="models/mnist.h5",
        session=sess,
        use_softmax=True
    )

    pred = sess.run(
        model.predict(
            data.test_data[:100]
        )
    )

    pred_labels = np.argmax(
        pred,
        axis=1
    )

    true_labels = np.argmax(
        data.test_labels[:100],
        axis=1
    )

    accuracy = np.mean(
        pred_labels == true_labels
    )

    print("Prediction shape:", pred.shape)
    print("Accuracy:", accuracy)
    print("First 20 predictions:", pred_labels[:20])
    print("First 20 labels:", true_labels[:20])

In [43]:
main_path = ROOT / "main.py"

text = main_path.read_text()

# Add TensorFlow import if missing.
if "import tensorflow as tf" not in text:
    text = "import tensorflow as tf\n" + text

# Ensure graph mode is enabled before model construction.
if "tf.compat.v1.disable_eager_execution()" not in text:
    text = (
        "import tensorflow as tf\n"
        "tf.compat.v1.disable_eager_execution()\n"
        "tf.compat.v1.reset_default_graph()\n\n"
        + text
    )
else:
    # Make sure reset happens after disable_eager_execution.
    pass

# Use the .h5 model.
text = text.replace(
    'MNISTModel("models/mnist", sess, use_softmax=True)',
    'MNISTModel("models/mnist.h5", sess, use_softmax=True)'
)

text = text.replace(
    "MNISTModel('models/mnist', sess, use_softmax=True)",
    "MNISTModel('models/mnist.h5', sess, use_softmax=True)"
)

main_path.write_text(text)

print("main.py patched.")

main.py patched.


In [32]:
from pathlib import Path

ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/"
    "BBAdvGen/Baselines/Autozoom-Attack"
)

utils_path = ROOT / "Utils.py"

text = utils_path.read_text()

old = "pred_logits = model.sess.run(\n        model.model.output,\n        feed_dict={\n            model.model.input: data.test_data\n        }\n    )\n    pred_labels = np.argmax(pred_logits, axis=1)"

new = """pred_logits = tf.compat.v1.get_default_session().run(
        model.predict(data.test_data)
    )
    pred_labels = np.argmax(pred_logits, axis=1)"""

if old not in text:
    print("Previous patch not found. Showing relevant lines:")
    for i, line in enumerate(text.splitlines(), 1):
        if 135 <= i <= 150:
            print(f"{i}: {line}")
else:
    utils_path.write_text(text.replace(old, new))
    print("Successfully patched Utils.py")

Successfully patched Utils.py


In [33]:
!python3 main.py -a zoo -d mnist -n 100 --m 1000 --batch_size 128 --switch_iterations 100 --init_const 10 --img_resize 14

/content/drive/.shortcut-targets-by-id/15IngWd3S2raLWMy55q5fIQL3J3MoNxbF/BBAdvGen/Baselines/Autozoom-Attack/main.py:27: DeprecationWarning: scipy.misc is deprecated and will be removed in 2.0.0
  import scipy.misc
{'attack_method': 'zoo', 'batch_size': 128, 'init_const': 10.0, 'dataset': 'mnist', 'num_img': 100, 'max_iterations': 1000, 'print_every': 100, 'save_path': 'Results', 'attack_type': 'targeted', 'confidence': 0, 'codec_prefix': 'codec/mnist', 'random_target': False, 'num_rand_vec': 1, 'seed': 9487, 'img_offset': 0, 'img_resize': 14, 'switch_iterations': 100, 'imagenet_dir': None, 'attack_single_img': None, 'single_img_target_label': None, 'compress_mode': 1, 'use_tanh': False, 'lr': 0.01}
2026-09-04 21:08:20.161357: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1788556100.162852    7867 gpu_device.cc:2020] Created device /jo